# HW3a - Build and Evaluate Classifiers

See Canvas for details on how to complete and submit this assignment.

## Introduction

This is Part 1 of HW3, which covers our classification unit (lectures 09a through 11b). In this part you'll build and evaluate classifiers on real manufacturing data. Parts 2 (paper reflection) and 3 (cost-sensitive evaluation) are released separately.

| Part | Topic | Release | Due | Points |
|------|-------|---------|-----|-------:|
| HW3a | Build & Evaluate Classifiers | Mar 17 | Mar 26 | 45 |
| HW3b | Paper: Cost-Sensitive Learning | Mar 17 | Mar 26 | 15 |
| HW3c | Cost-Sensitive Evaluation | Mar 24 | Apr 6 | 40 |
| | Total | | | 100 |

### Learning Objectives

By completing this part, you will demonstrate ability to:

- Build classification Pipelines with proper preprocessing
- Evaluate classifiers using precision, recall, F1, and confusion matrices
- Use `GridSearchCV` for systematic hyperparameter tuning
- Compare models using a summary table and interpret the gaps between them

### Time Estimate

This part should take ~2 hours to complete.

### Generative AI Allowance

You may use GenAI tools for brainstorming, explanations, and code sketches if you disclose it, understand it, and validate it. Your submission must represent your own work and you are solely responsible for its correctness.

### Scoring

| Section | Points |
|---------|-------:|
| Exploration | 5 |
| Logistic Regression Pipeline | 15 |
| KNN Classifier | 15 |
| Model Comparison | 10 |
| **Total** | **45** |

## Background

You work as a quality engineer at a steel manufacturing plant. Your task is to build a classifier that identifies the type of surface defect on steel plates based on sensor and geometry measurements. Accurate fault classification helps route defective plates to the right repair process, reducing waste and downtime.

The Steel Plates Faults dataset (UCI Machine Learning Repository) contains 1,941 observations with 27 numeric features describing the geometry, luminosity, and physical properties of steel plate defects. There are 7 fault types: Pastry, Z_Scratch, K_Scratch, Stains, Dirtiness, Bumps, and Other_Faults.

## Setup

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Load and Explore (5 pts)

Load the Steel Plates Faults dataset using `fetch_openml`. The dataset ID is 40982.

In [ ]:
# Load dataset
steel = fetch_openml(data_id=40982, as_frame=True, parser="auto")
X, y = steel.data, steel.target

print(f"Shape: {X.shape}")
print(f"Classes: {y.nunique()}")
print(f"\nClass distribution:")
print(y.value_counts())

In [ ]:
# Plot the class distribution as a bar chart
fig, ax = plt.subplots(figsize=(8, 4))
y.value_counts().plot.bar(ax=ax, color="steelblue", edgecolor="black")
ax.set_xlabel("Fault Type")
ax.set_ylabel("Count")
ax.set_title("Steel Plates Faults - Class Distribution")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Split into training and test sets. Use `test_size=0.3` and `random_state=42`. Some fault types are rare (Dirtiness is only ~3% of the data), so use the `stratify` parameter to preserve the class distribution in both sets.

In [ ]:
# Split into train/test sets
# Use test_size=0.3, random_state=42, and stratify on the target
# See 09a notebook for an example of stratified splitting

# Your code here...

print(f"Training set: {X_train.shape[0]} observations, {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]} observations")

---

##### Exploration Questions

1. Which fault type is most common? Which is rarest? What is the ratio between them?
2. Why does the `stratify` parameter matter for this dataset? What could go wrong without it, given the class sizes you see above?

*Your answers here...*

---

## Logistic Regression Pipeline (15 pts)

Build a Pipeline with `StandardScaler` and `LogisticRegression`. Feature scales in this dataset vary enormously - some features have means near zero while others are in the millions - so scaling is essential.

Use `LogisticRegression(max_iter=5000)` to ensure convergence. The default solver (`lbfgs`) uses the multinomial loss automatically for multiclass problems.

In [ ]:
# Build and fit a classification Pipeline: StandardScaler → LogisticRegression
# See 09a notebook, Section 3 for an example of a classification Pipeline
# Store the fitted pipeline in a variable called logreg_pipe

# Your code here...

In [ ]:
# Print training and test accuracy using .score()
# Then print classification_report for the test set
# You'll need predictions from .predict() for the classification_report

# Your code here...

In [ ]:
# Plot the confusion matrix
# ConfusionMatrixDisplay.from_estimator(model, X_test, y_test) does it in one call
fig, ax = plt.subplots(figsize=(8, 6))

# Your code here (one line)...

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

---

##### Logistic Regression Interpretation

1. Look at the per-class precision, recall, and support in the classification report. Which fault types does the model handle well? Which does it struggle with? Does the support column (number of test samples per class) help explain any of the differences?
2. In the confusion matrix, identify the two most common misclassifications (off-diagonal cells with the highest counts). Why might those fault types be confused with each other?

*Your answers here...*

---

## KNN Classifier (15 pts)

In HW2 you used KNN for regression with manual cross-validation loops. `GridSearchCV` automates this: it searches over a grid of hyperparameter values, evaluates each with cross-validation, and selects the best. You'll use it here for KNN and again in HW3c for SVM.

*Before fitting*: how do you expect KNN to compare with logistic regression on this dataset? Consider that logistic regression draws linear boundaries between classes, while KNN draws flexible boundaries that adapt to local structure.

---

##### Prediction

*Your prediction and reasoning here...*

---

Use `GridSearchCV` to find the best k for KNN classification. The key ideas:

- Build a Pipeline with `StandardScaler` and `KNeighborsClassifier`
- Define a `param_grid` dict mapping parameter names to values to search
- Parameter names use `stepname__param` format (e.g., `"knn__n_neighbors"`)
- `GridSearchCV` fits and scores every combination using cross-validation

In [ ]:
# Use GridSearchCV to tune n_neighbors for KNN
# Build a Pipeline: StandardScaler → KNeighborsClassifier
# Search over n_neighbors: [1, 3, 5, 10, 20, 50]
# Use cv=5 and scoring="accuracy"
# Use return_train_score=True so we can plot training vs CV accuracy

# Your code here... (store the fitted GridSearchCV in a variable called knn_search)

print(f"Best k: {knn_search.best_params_['knn__n_neighbors']}")
print(f"Best CV accuracy: {knn_search.best_score_:.3f}")

In [ ]:
# Plot training accuracy and CV accuracy vs. k
# GridSearchCV stores results in cv_results_ - we extract what we need
results = pd.DataFrame(knn_search.cv_results_)
k_values = results["param_knn__n_neighbors"].astype(int)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_values, results["mean_train_score"], "o-", label="Training")
ax.plot(k_values, results["mean_test_score"], "s-", label="CV (mean)")
ax.set_xlabel("k (number of neighbors)")
ax.set_ylabel("Accuracy")
ax.set_title("KNN Performance vs. k")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate the best KNN model on the test set
# knn_search.predict() uses the best model automatically
# Print test accuracy, classification_report, and store best k for later use
best_k = knn_search.best_params_["knn__n_neighbors"]

# Your code here...

---

##### KNN Interpretation

1. Describe the shape of the training accuracy curve as k increases. Why does k=1 give perfect (or near-perfect) training accuracy? Is this a good thing?
2. How did KNN compare to logistic regression? What does this suggest about the class boundaries in this dataset - mostly linear, or does local structure matter?

*Your answers here...*

---

## Model Comparison (10 pts)

Build a summary table comparing all models. The table includes a `DummyClassifier` baseline, both multinomial and One-vs-Rest logistic regression, and your best KNN model.

*Macro F1* averages F1 across all classes equally, giving rare classes like Dirtiness (55 samples) the same weight as Other_Faults (673 samples). *Weighted F1* weights by class size, so it's dominated by the larger classes. For a quality engineering problem where rare faults matter, think about which average is more informative.

`OneVsRestClassifier` wraps a binary classifier to handle multiclass problems by fitting one model per class. It's an alternative to the default multinomial approach. See `sklearn.multiclass.OneVsRestClassifier` in the docs.

In [ ]:
models = {
    "Dummy": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", DummyClassifier(strategy="most_frequent")),
    ]),
    "LogReg (multinomial)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000)),
    ]),
    # Add a LogReg (OvR) entry using OneVsRestClassifier wrapping LogisticRegression
    # See the OneVsRestClassifier import at the top of the notebook

    # Your code here...

    # Add a KNN entry using best_k from the previous section

    # Your code here...
}

rows = []
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy")
    rows.append({
        "Model": name,
        "Train Acc": pipe.score(X_train, y_train),
        "Test Acc": accuracy_score(y_test, y_pred),
        "CV Acc (mean)": cv.mean(),
        "CV Acc (std)": cv.std(),
        "Macro F1": f1_score(y_test, y_pred, average="macro"),
        "Weighted F1": f1_score(y_test, y_pred, average="weighted"),
    })

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))

In [ ]:
# Plot a normalized confusion matrix for the best model
fig, ax = plt.subplots(figsize=(8, 6))
best_model_name = comparison_df.loc[comparison_df["Test Acc"].idxmax(), "Model"]
best_model = models[best_model_name]
ConfusionMatrixDisplay.from_estimator(
    best_model, X_test, y_test, normalize="true", ax=ax, values_format=".2f"
)
ax.set_title(f"Normalized Confusion Matrix - {best_model_name}")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

---

##### Model Comparison Analysis

1. Interpret the gaps in the comparison table:
   - Dummy → LogReg: how much does a real model improve over the naive baseline?
   - LogReg → KNN: does a flexible decision boundary help? What does this reveal about the data?
   - Multinomial vs OvR: how do they compare? Is the difference meaningful?
2. Look at the normalized confusion matrix for your best model. Which fault types are classified most reliably? Which have the lowest recall? If you were the quality engineer, which misclassifications would concern you most and why?

*Your answers here...*

---